# Train a tiny LLM on SEC filings

Run this notebook **on your laptop** (in WSL or any local Jupyter). It uses your RunPod API key to spin up a 1× A100 80 GB pod, uploads SEC filing data from `data/filings-2025-2026/`, runs Karpathy's [nanochat](https://github.com/karpathy/nanochat) training pipeline on the pod, and lets you chat with the result.

Total time: ~10 min (pod boot + data upload + ~5 min training).

**Before running:**

```bash
pip install runpod requests
# WSL/Linux/Mac: ssh + scp + ssh-keygen come pre-installed.
```

Run cells top to bottom. The last cell terminates the pod — don't skip it.

## 1. Paste your RunPod API key

Generate one at https://www.runpod.io/console/user/settings (you also need at least \$10 of credit; the workshop run is \$0.30–0.50).

In [1]:
import getpass, runpod
runpod.api_key = getpass.getpass("RunPod API key: ")
me = runpod.get_user()
print(f"OK, logged in as {me.get('email', '?')}")

RunPod API key:  ········


OK, logged in as ?


## 2. Constants

Tweak only if you want to change which SEC sections to upload, or push the model bigger. Defaults: just the 12 MB `market_risk` section, depth-4 GPT (~5 M params), 400 training steps. Fits in ~10 min wall clock.

In [ ]:
import pathlib

DATA_DIR    = pathlib.Path("data/filings-2025-2026").resolve()

# Which parquets to upload (smallest first). Each row is one SEC filing section.
PARQUETS = [
    "market_risk_2025_2026.parquet",   # 12 MB — uploads in seconds
    # "mda_2025_2026.parquet",         # 207 MB — uncomment for richer corpus
    # "business_2025_2026.parquet",    # 290 MB
    # "risk_factors_2025_2026.parquet",# 470 MB — most stylistically distinctive
]

POD_NAME    = "zero-to-llm"

# GPUs to try, best-fit-first. The model is tiny (~5 M params) so anything
# >=24 GB VRAM is plenty. RunPod's `get_gpus()` lists what their catalog
# *contains*, not what's allocatable *right now*, so the next cell walks
# this list and try-creates each pod until one succeeds.
GPU_PREFS = [
    "NVIDIA A100 80GB PCIe",
    "NVIDIA A100-SXM4-80GB",
    "NVIDIA H100 80GB HBM3",
    "NVIDIA H100 PCIe",
    "NVIDIA H100 NVL",
    "NVIDIA L40S",
    "NVIDIA L40",
    "NVIDIA RTX A6000",
    "NVIDIA GeForce RTX 4090",
    "NVIDIA RTX 6000 Ada Generation",
    "NVIDIA RTX A5000",
    "NVIDIA GeForce RTX 3090",
]
POD_IMAGE   = "runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04"
DISK_GB     = 60

# Training shape — these are passed to nanochat's scripts/base_train.py
DEPTH         = 4         # 4 -> ~5 M params, 6 -> ~14 M, 8 -> ~30 M
NUM_STEPS     = 400       # ~5–7 min on A100 at depth=4
VOCAB_SIZE    = 4096
MAX_SEQ_LEN   = 1024
DEV_BATCH     = 8
TOTAL_BATCH   = 32768
TOK_MAX_CHARS = 20_000_000

print(f"Will upload {len(PARQUETS)} parquet(s) totaling "
      f"{sum((DATA_DIR/p).stat().st_size for p in PARQUETS)/1e6:.1f} MB")

## 3. Make sure SSH key exists & is registered with RunPod

If you don't have one at `~/.ssh/id_ed25519`, this generates one (no passphrase) and uploads the public part to your RunPod account. RunPod injects it into the pod's `authorized_keys` automatically.

In [3]:
import subprocess, pathlib

priv = pathlib.Path.home() / ".ssh" / "id_ed25519"
pub  = priv.with_suffix(".pub")
if not priv.exists():
    print(f"Generating SSH key at {priv} ...")
    priv.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-f", str(priv), "-N", "", "-q"], check=True)

pubkey = pub.read_text().strip()
runpod.update_user_settings(pubkey=pubkey)
print(f"Registered public key with RunPod ({pubkey[:50]}...)")

Generating SSH key at /home/neilcelik/.ssh/id_ed25519 ...
Registered public key with RunPod (ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIOKUQ0CXXiYbX...)


## 4. Spin up the GPU pod

Picks the first available GPU from the preference list. The pod's default entrypoint runs JupyterLab in the background — we ignore it and SSH in directly for everything.

In [ ]:
import time
from runpod.error import QueryError

available = {g['id'] for g in runpod.get_gpus()}
candidates = [g for g in GPU_PREFS if g in available]
print(f"Catalog candidates ({len(candidates)}): {candidates}")
if not candidates:
    raise RuntimeError(f"None of GPU_PREFS appear in the RunPod catalog. Catalog: {sorted(available)}")

# Walk the candidates list and try-create each pod until one succeeds.
# We have to actually attempt the create — `get_gpus()` only tells us what
# *exists* in the catalog, not what's *allocatable* right now.
pod = None
gpu_type = None
for gpu in candidates:
    try:
        print(f"Trying {gpu} ...")
        pod = runpod.create_pod(
            name=POD_NAME,
            image_name=POD_IMAGE,
            gpu_type_id=gpu,
            gpu_count=1,
            cloud_type='ALL',
            container_disk_in_gb=DISK_GB,
            ports='22/tcp,8888/http',     # 22 for our SSH, 8888 for the optional JupyterLab UI
            support_public_ip=True,
            start_ssh=True,
            # NOTE: do NOT pass docker_args. The runpod/pytorch image's default
            # entrypoint starts sshd; overriding it kills sshd before it ever runs.
        )
        gpu_type = gpu
        break
    except QueryError as e:
        # "no instances available" / "out of capacity" — try next candidate
        msg = str(e).lower()
        if 'no longer any instances' in msg or 'no instances available' in msg or 'out of capacity' in msg:
            print(f"  out of stock, trying next ...")
            continue
        raise

if pod is None:
    raise RuntimeError("Every GPU type in GPU_PREFS is out of stock right now. Wait a few minutes and re-run this cell.")
print(f"Got pod on {gpu_type}")
POD_ID = pod['id']
print(f"Pod created: {POD_ID}")

# Wait for runtime info (public SSH IP+port)
SSH_HOST = SSH_PORT = None
for _ in range(60):
    info = runpod.get_pod(POD_ID)
    for p in (info.get('runtime') or {}).get('ports', []):
        if p.get('privatePort') == 22 and p.get('isIpPublic'):
            SSH_HOST, SSH_PORT = p['ip'], p['publicPort']
            break
    if SSH_HOST: break
    time.sleep(5)
    print(f"  ... waiting for SSH endpoint")

if not SSH_HOST:
    raise RuntimeError("Pod never exposed a public SSH endpoint. Check the RunPod console.")
print(f"SSH endpoint: root@{SSH_HOST}:{SSH_PORT}")

## 5. Wait for SSH to actually accept connections

Above, the *endpoint* exists. Now we wait until sshd inside the container is taking connections (~30–60 s).

In [ ]:
import subprocess

SSH_BASE = ['ssh', '-p', str(SSH_PORT),
            '-o', 'StrictHostKeyChecking=no',
            '-o', 'UserKnownHostsFile=/dev/null',
            '-o', 'LogLevel=ERROR',
            '-o', 'ConnectTimeout=5',
            f'root@{SSH_HOST}']
SCP_BASE = ['scp', '-P', str(SSH_PORT),
            '-o', 'StrictHostKeyChecking=no',
            '-o', 'UserKnownHostsFile=/dev/null',
            '-o', 'LogLevel=ERROR']

t0 = time.time()
for attempt in range(60):
    rc = subprocess.run(SSH_BASE + ['true'], stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL).returncode
    if rc == 0:
        print(f"SSH up after {int(time.time()-t0)}s")
        break
    time.sleep(5)
    print(f"  ... waiting for sshd ({int(time.time()-t0)}s)")
else:
    raise RuntimeError("SSH never accepted connections. Check pod logs in the RunPod console.")

## 6. Helper: stream output from remote commands

Two small wrappers over `subprocess`:
- `ssh_run(cmd)` — runs `cmd` on the pod, streams stdout to this notebook line by line, raises on nonzero exit.
- `scp_upload(local, remote)` — copies one file from your laptop to the pod.

In [ ]:
import subprocess, pathlib

def ssh_run(cmd, stream=True):
    proc = subprocess.Popen(SSH_BASE + [cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        if stream: print(line, end='', flush=True)
        out.append(line)
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"remote command failed (rc={proc.returncode}): {cmd[:100]}...")
    return ''.join(out)

def scp_upload(local, remote):
    subprocess.run(SCP_BASE + [str(local), f'root@{SSH_HOST}:{remote}'], check=True)
    print(f"  uploaded {pathlib.Path(local).name} -> {remote}")

## 7. Upload the SEC parquets to the pod

In [ ]:
ssh_run("mkdir -p /workspace/sec_data")
for fname in PARQUETS:
    src = DATA_DIR / fname
    if not src.exists():
        raise FileNotFoundError(src)
    print(f"Uploading {fname} ({src.stat().st_size/1e6:.1f} MB)...")
    scp_upload(src, f"/workspace/sec_data/{fname}")
print("done.")

## 8. Install nanochat on the pod

Clones karpathy/nanochat, installs deps with `uv` (fast, ~1 min on A100 box), and stages the SEC parquets into nanochat's expected `~/.cache/nanochat/base_data_climbmix/` layout (one shard per parquet plus a final val shard).

In [ ]:
INSTALL = '''
set -e
# Triton's torch.compile path needs Python dev headers + a working gcc to
# build cuda_utils.c during the first training step. The runpod/pytorch
# image doesn't ship them by default.
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq python3.10-dev python3.11-dev build-essential
cd /workspace
test -d nanochat || git clone https://github.com/karpathy/nanochat.git
curl -LsSf https://astral.sh/uv/install.sh | sh >/dev/null 2>&1
export PATH="$HOME/.local/bin:$PATH"
cd nanochat
uv venv
uv sync --extra gpu
'''
ssh_run(INSTALL)

In [ ]:
# Stage SEC parquets into nanochat's data dir.
# nanochat reads ~/.cache/nanochat/base_data_climbmix/shard_NNNNN.parquet
# (last shard = val). We re-export each input parquet keeping only `text`,
# split into N+1 shards (last one held out for val).
PREP = '''
set -e
source /workspace/nanochat/.venv/bin/activate
python - <<'PY'
import pathlib, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, os
src_dir = pathlib.Path("/workspace/sec_data")
target = pathlib.Path(os.path.expanduser("~/.cache/nanochat/base_data_climbmix"))
target.mkdir(parents=True, exist_ok=True)
for old in target.glob("shard_*.parquet"): old.unlink()

frames = []
for p in sorted(src_dir.glob("*.parquet")):
    df = pd.read_parquet(p, columns=["text"])
    df = df[df["text"].str.len() > 200].reset_index(drop=True)
    print(f"  {p.name}: {len(df):,} docs after filtering")
    frames.append(df)
big = pd.concat(frames, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Total docs: {len(big):,}")

NUM = 4
sz = (len(big) + NUM - 1) // NUM
for i in range(NUM):
    chunk = big.iloc[i*sz:(i+1)*sz]
    if len(chunk) == 0: continue
    out = target / f"shard_{i:05d}.parquet"
    pq.write_table(pa.Table.from_pandas(chunk[["text"]]), out)
    print(f"  wrote {out.name} rows={len(chunk):,} size={out.stat().st_size/1e6:.1f}MB")
PY
'''
ssh_run(PREP)

## 9. Train a custom BPE tokenizer (~30 s)

This trains a small BPE tokenizer (vocab 4096) on ~20 M chars of SEC text. Identical to nanochat's `scripts/tok_train.py`, just with smaller `--max-chars` and `--vocab-size`.

In [ ]:
ssh_run(f'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.tok_train --max-chars {TOK_MAX_CHARS} --vocab-size {VOCAB_SIZE}
''')

## 10. Pretrain the GPT (~5–7 min)

The actual training run. You'll see `step N/{NUM_STEPS} | loss=... | tok/s=...` lines streaming in. Loss should drop from ~7-8 down to ~3-4.

In [ ]:
ssh_run(f'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.base_train \\
    --depth {DEPTH} \\
    --max-seq-len {MAX_SEQ_LEN} \\
    --device-batch-size {DEV_BATCH} \\
    --total-batch-size {TOTAL_BATCH} \\
    --num-iterations {NUM_STEPS} \\
    --window-pattern L \\
    --eval-tokens 4096 \\
    --eval-every 100 \\
    --sample-every 100 \\
    --core-metric-every -1 \\
    --run dummy
''')

## 11. Sample generations from your model

Loads the just-saved checkpoint and prints completions for a few SEC-flavored prompts.

⚠️ This is a **base** model (no instruction tuning). Treat each prompt as the *start* of a passage that the model continues — not a question.

In [ ]:
SAMPLE = r'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python - <<'PY'
import os
os.environ.setdefault("MASTER_ADDR","localhost"); os.environ.setdefault("MASTER_PORT","29500")
os.environ.setdefault("RANK","0"); os.environ.setdefault("WORLD_SIZE","1"); os.environ.setdefault("LOCAL_RANK","0")
from nanochat.checkpoint_manager import load_model
from nanochat.engine import Engine
from nanochat.common import autodetect_device_type, compute_init
dt = autodetect_device_type()
_,_,_,_,device = compute_init(dt)
model, tok, meta = load_model("base", device, phase="eval")
engine = Engine(model, tok)

PROMPTS = [
    "ITEM 1A. RISK FACTORS\n\nThe following risks could materially affect our",
    "Our business is focused on",
    "We may be unable to",
    "Management's Discussion and Analysis of Financial Condition\n\nOverview:",
]
for p in PROMPTS:
    print("\n" + "-"*70)
    print("Prompt:", repr(p)); print("Continuation:")
    print(p, end="")
    toks = [tok.get_bos_token_id()] + tok.encode(p)
    for tc, _ in engine.generate(toks, num_samples=1, max_tokens=120, temperature=0.8, top_k=50):
        print(tok.decode([tc[0]]), end="", flush=True)
    print()
PY
'''
ssh_run(SAMPLE)

## 12. Chat with your model

`chat(prompt)` sends `prompt` to the pod, generates a continuation, and returns it. Edit and re-run the cell with whatever prompt you like.

Each call reloads the model on the pod (~5 s overhead). Good enough for a few demo prompts; if you want a faster interactive loop, you'd start an inference server on the pod, but that's out of scope here.

In [ ]:
import shlex

def chat(prompt: str, max_tokens: int = 200, temperature: float = 0.8, top_k: int = 50) -> str:
    """Generate a continuation of `prompt` on the trained model."""
    py_prompt = prompt.replace("\\", "\\\\").replace("'", "\\'")
    code = f'''
import os
os.environ.setdefault("MASTER_ADDR","localhost"); os.environ.setdefault("MASTER_PORT","29500")
os.environ.setdefault("RANK","0"); os.environ.setdefault("WORLD_SIZE","1"); os.environ.setdefault("LOCAL_RANK","0")
from nanochat.checkpoint_manager import load_model
from nanochat.engine import Engine
from nanochat.common import autodetect_device_type, compute_init
dt = autodetect_device_type()
_,_,_,_,device = compute_init(dt)
model, tok, meta = load_model("base", device, phase="eval")
engine = Engine(model, tok)
prompt = {prompt!r}
toks = [tok.get_bos_token_id()] + tok.encode(prompt)
for tc, _ in engine.generate(toks, num_samples=1, max_tokens={max_tokens}, temperature={temperature}, top_k={top_k}):
    print(tok.decode([tc[0]]), end="", flush=True)
print()
'''
    full = (
        "source /workspace/nanochat/.venv/bin/activate && "
        "cd /workspace/nanochat && "
        f"python -c {shlex.quote(code)}"
    )
    print(prompt, end="")
    out = ssh_run(full)
    return out

_ = chat("ITEM 1A. RISK FACTORS\n\nThe primary risks include")

In [ ]:
# Try your own prompt — edit and re-run.
_ = chat("Our principal sources of revenue are")

## 13. Embed each company-year using the trained model

We'll use the GPT we just trained as an embedder: feed each (ticker, year) filing through the model, mean-pool the last layer's hidden states across tokens, and that 256-dim vector is our embedding.

Two caveats up front:
- This is a **5 M-param model trained for ~5 min** on a tiny corpus. Its embeddings will capture surface stylistic features (vocabulary, document length, recurring SEC phrases) more than deep semantic meaning. Don't expect the same quality as `sentence-transformers` or OpenAI embeddings.
- The model's max context is 1024 tokens, but a full 4-section concatenation per company-year is much longer. We chunk the text into 1024-token windows, embed each chunk, and average across chunks.

Before running these cells you need all four parquets on the pod (the earlier cells only uploaded `market_risk_2025_2026.parquet`). The next cell uploads the missing three (~1 GB total, ~3 min on a typical home connection).

In [ ]:
# Upload the remaining 3 parquets so we have all 4 sections per company-year.
import os
ALL_PARQUETS = [
    "business_2025_2026.parquet",
    "market_risk_2025_2026.parquet",
    "mda_2025_2026.parquet",
    "risk_factors_2025_2026.parquet",
]
ssh_run("mkdir -p /workspace/sec_data_full")
for fname in ALL_PARQUETS:
    src = DATA_DIR / fname
    if not src.exists():
        raise FileNotFoundError(src)
    size_mb = src.stat().st_size / 1e6
    print(f"Uploading {fname} ({size_mb:.1f} MB)...")
    scp_upload(src, f"/workspace/sec_data_full/{fname}")
print("done.")

### 13.1 Compute embeddings on the pod

This runs entirely on the pod. It:
1. Joins all four parquets into one DataFrame keyed by `(ticker, year)`, concatenating the text from each section.
2. For each (ticker, year), tokenizes the concatenated text with our trained BPE tokenizer.
3. Splits into 1024-token chunks, runs each chunk through the model, mean-pools each chunk's last-layer hidden state, and averages the chunk vectors.
4. Saves the result to `/workspace/embeddings.parquet` with columns `ticker, year, num_tokens, embedding`.

On an A100, embedding ~3 K (ticker, year) pairs with ~30 chunks each takes about 30–60 s.

In [ ]:
EMBED = r'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python - <<'PY'
import os, time, pathlib, numpy as np, pandas as pd, torch
os.environ.setdefault("MASTER_ADDR","localhost"); os.environ.setdefault("MASTER_PORT","29500")
os.environ.setdefault("RANK","0"); os.environ.setdefault("WORLD_SIZE","1"); os.environ.setdefault("LOCAL_RANK","0")
from nanochat.checkpoint_manager import load_model
from nanochat.common import autodetect_device_type, compute_init

# 1) Load all four parquets, derive year from filing_date.
print("Loading parquets...")
dfs = []
for f in sorted(pathlib.Path("/workspace/sec_data_full").glob("*.parquet")):
    df = pd.read_parquet(f, columns=["ticker", "filing_date", "section_name", "text"])
    df = df[df["text"].str.len() > 200]
    df["year"] = pd.to_datetime(df["filing_date"]).dt.year
    dfs.append(df)
big = pd.concat(dfs, ignore_index=True)
print(f"  total rows: {len(big):,}")

# 2) Group by (ticker, year), concat all section text per group.
SEP = chr(10) + chr(10)
print("Grouping by (ticker, year)...")
grouped = (big.sort_values(["ticker", "year", "section_name"])
              .groupby(["ticker", "year"])["text"]
              .apply(lambda s: SEP.join(s))
              .reset_index())
n_docs = len(grouped)
print(f"  unique (ticker, year): {n_docs:,}")

# 3) Load model + tokenizer.
dt = autodetect_device_type()
_,_,_,_,device = compute_init(dt)
model, tok, meta = load_model("base", device, phase="eval")
seq_len = model.config.sequence_len
n_embd = model.config.n_embd
print(f"Model: seq_len={seq_len}, n_embd={n_embd}")

# 4) Hook the final transformer block to capture hidden states pre-LM-head.
captured = {}
def _hook(module, inp, out):
    captured["h"] = out if isinstance(out, torch.Tensor) else out[0]
hook = model.transformer.h[-1].register_forward_hook(_hook)

# 5) Tokenize EVERY (ticker, year) once, build a flat (doc_idx, chunk) list.
print("Tokenizing...")
t0 = time.time()
ntoks_per_doc = []
flat_chunks = []   # list of (doc_idx, list_of_token_ids)
for doc_idx, txt in enumerate(grouped["text"].values):
    ids = tok.encode(txt)
    ntoks_per_doc.append(len(ids))
    for s in range(0, len(ids), seq_len):
        ch = ids[s:s+seq_len]
        if len(ch) >= 8:                  # skip useless tails
            flat_chunks.append((doc_idx, ch))
print(f"  tokenized in {time.time()-t0:.1f}s; total chunks: {len(flat_chunks):,}")

# Sort by chunk length descending so batches have similar-length sequences (less padding waste).
flat_chunks.sort(key=lambda x: -len(x[1]))

# 6) Process in batches; sum chunk-means per (ticker, year), then average at the end.
B = 64                                    # batch size; model is 5M params, room for plenty
sum_per_doc   = np.zeros((n_docs, n_embd), dtype=np.float32)
count_per_doc = np.zeros(n_docs, dtype=np.int32)

print(f"Embedding in batches of {B}...")
t0 = time.time()
with torch.inference_mode():
    for i in range(0, len(flat_chunks), B):
        batch = flat_chunks[i:i+B]
        bsz   = len(batch)
        mlen  = max(len(c[1]) for c in batch)
        padded = torch.zeros((bsz, mlen), dtype=torch.long, device=device)
        lens   = torch.empty(bsz, dtype=torch.long)
        for j, (_, ch) in enumerate(batch):
            padded[j, :len(ch)] = torch.tensor(ch, device=device)
            lens[j] = len(ch)
        _ = model(padded)
        h = captured["h"].float()         # (B, mlen, n_embd)
        # Mean-pool each row over its actual length.
        for j, (doc_idx, _) in enumerate(batch):
            L = int(lens[j])
            sum_per_doc[doc_idx] += h[j, :L].mean(dim=0).cpu().numpy()
            count_per_doc[doc_idx] += 1
        if (i // B) % 20 == 0:
            done = i + bsz
            rate = done / max(time.time() - t0, 1e-6)
            eta  = (len(flat_chunks) - done) / max(rate, 1e-6)
            print(f"  {done:>6}/{len(flat_chunks)}  ({rate:.0f} chunks/s, ETA {eta:.0f}s)")
hook.remove()
print(f"  embedding done in {time.time()-t0:.1f}s")

# 7) Average chunk-means per doc.
embeddings = np.zeros((n_docs, n_embd), dtype=np.float32)
nonzero = count_per_doc > 0
embeddings[nonzero] = sum_per_doc[nonzero] / count_per_doc[nonzero, None]

# 8) Save.
out = pd.DataFrame({
    "ticker": grouped["ticker"].values,
    "year":   grouped["year"].values,
    "num_tokens": ntoks_per_doc,
    "embedding": list(embeddings),        # one np.float32 vector per row
})
out_path = pathlib.Path("/workspace/embeddings.parquet")
out.to_parquet(out_path, index=False)
print(f"Saved {len(out):,} rows -> {out_path} ({out_path.stat().st_size/1e6:.1f} MB)")
PY
'''
ssh_run(EMBED)

### 13.2 Download the embeddings to your laptop

In [ ]:
import subprocess, pathlib

# Pull /workspace/embeddings.parquet down to ./embeddings.parquet
local_path = pathlib.Path("embeddings.parquet").resolve()
subprocess.run(SCP_BASE + [f"root@{SSH_HOST}:/workspace/embeddings.parquet", str(local_path)], check=True)
print(f"Downloaded -> {local_path} ({local_path.stat().st_size/1e6:.1f} MB)")

## 14. Analyze the embeddings (locally)

These cells run on your laptop, on the small downloaded `embeddings.parquet`. We'll need a few extra Python packages — the next cell installs them into the local venv. (`scikit-learn` for PCA and k-means; `matplotlib` for plots; `pandas`+`pyarrow` for reading the parquet.)

In [ ]:
# One-time install of analysis deps. Skips packages already present, and
# retries with --break-system-packages on PEP 668 failures (Ubuntu 22.04+
# system Python rejects unmanaged pip installs without that flag).
import sys, subprocess, importlib

REQUIRED = {                       # import name -> pip name
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
}
missing = [pip_name for mod, pip_name in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print(f"Installing: {missing}")
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        print("First attempt failed; retrying with --break-system-packages "
              "(your Python is system-managed; PEP 668).")
        subprocess.check_call(cmd + ["--break-system-packages"])

import pandas as pd, numpy as np
df = pd.read_parquet("embeddings.parquet")
X = np.stack(df["embedding"].values).astype(np.float32)   # (N, 256)
print(f"Loaded {len(df):,} embeddings, shape={X.shape}, dtype={X.dtype}")
df.head()

### 14.1 PCA scatter

Project 256 dims down to 2 via PCA (linear, fast, deterministic). Color by filing year. If the trained model picked up on time-related signals, you'll see year-based clustering.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=0)
xy = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=df["year"], s=8, alpha=0.6, cmap="viridis")
plt.colorbar(sc, ax=ax, label="year")
ax.set_title(f"PCA of (ticker, year) embeddings  |  explained var: "
             f"{pca.explained_variance_ratio_.sum():.2%}")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

### 14.2 t-SNE scatter (slower, often more visually separated)

t-SNE is a non-linear projection that often makes clusters look more distinct than PCA. It's slower (~10–30 s for ~3 K points) and non-deterministic.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=0)
xy_t = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(xy_t[:, 0], xy_t[:, 1], c=df["year"], s=8, alpha=0.6, cmap="viridis")
plt.colorbar(sc, ax=ax, label="year")
ax.set_title("t-SNE of (ticker, year) embeddings")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
plt.tight_layout(); plt.show()

### 14.3 Nearest-neighbor lookup

`find_similar(ticker, year, k=5)` returns the k closest (ticker, year) pairs by cosine similarity. Try a ticker you know — what shows up?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Pre-normalize once
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

def find_similar(ticker: str, year: int, k: int = 5) -> pd.DataFrame:
    idx = df.index[(df["ticker"] == ticker) & (df["year"] == year)].tolist()
    if not idx:
        avail = df[df["ticker"] == ticker]["year"].tolist()
        raise KeyError(f"({ticker}, {year}) not found. Years for {ticker}: {avail}")
    i = idx[0]
    sims = (Xn @ Xn[i])
    order = np.argsort(-sims)
    order = [j for j in order if j != i][:k]
    return pd.DataFrame({
        "ticker": df["ticker"].iloc[order].values,
        "year":   df["year"].iloc[order].values,
        "cos_sim": sims[order],
    })

# Pick any (ticker, year) that exists in the data — first one is a safe default.
example_ticker = df.iloc[0]["ticker"]
example_year   = int(df.iloc[0]["year"])
print(f"5 nearest to ({example_ticker}, {example_year}):")
find_similar(example_ticker, example_year, k=5)

### 14.4 K-means clustering

Cluster the embeddings into K groups; print a few representative tickers for each cluster (the points closest to each cluster centroid). This is the model's attempt at finding "themes" in the SEC corpus.

In [ ]:
from sklearn.cluster import KMeans

K = 10
km = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = km.fit_predict(X)
df["cluster"] = labels

# For each cluster, take the 5 points closest to the centroid.
print(f"K={K} clusters; 5 representative (ticker, year) per cluster:\n")
for c in range(K):
    members = np.where(labels == c)[0]
    if len(members) == 0:
        continue
    centroid = km.cluster_centers_[c]
    dists = np.linalg.norm(X[members] - centroid, axis=1)
    closest = members[np.argsort(dists)[:5]]
    rows = df.iloc[closest][["ticker", "year"]].values.tolist()
    formatted = ", ".join(f"{t}({y})" for t, y in rows)
    print(f"  cluster {c:2d}  n={len(members):4d}  -> {formatted}")

## 15. Terminate the pod

⚠️ **Don't skip this.** Idle GPU pods cost real money.

In [ ]:
runpod.terminate_pod(POD_ID)
print(f"Terminated pod {POD_ID}.")